In [44]:
import sqlalchemy
import pymysql
import importlib
from dotenv import load_dotenv
from pathlib import Path

from sqlalchemy import create_engine, text
import pandas as pd
import os
import yaml
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent
import functions as fn
with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
             
#Data raw folder path:
raw_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\raw"
#Data clean folder path:
clean_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\clean"



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
importlib.reload(fn)

<module 'functions' from 'C:\\Users\\ziden\\Desktop\\Trainings\\RNCP-Project\\functions.py'>

In [36]:
#-----------------------------------------------------------------------------
# 1. ESTABLISH SQL CONNECTION AND UPLOAD A TABLE : function name: load_to_sql(file_to_load, sql_table_name)
#-----------------------------------------------------------------------------

#function name: load_to_sql(file_to_load, sql_table_name)
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(connection_string)

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT DATABASE();"))
        print(f"Connected to SQL DB: {result.scalar()}")
except Exception as err:
    print(f"Failed to connect to MySQL: {err}")

        


Connected to SQL DB: mobile_5g_network_fr


In [72]:
#-----------------------------------------------------------------------------
# 2. LOADING insee_geo_dfs to SQL
#-----------------------------------------------------------------------------
insee_geo_df = fn.insee_geo_flat_extract()

fn.load_to_sql(file_to_load=insee_geo_df, sql_table_name="dim_geo")


File 'insee_geo.csv' is successfully saved to 'data/clean' folder.
Connected to SQL DB: mobile_5g_network_fr
Uploading 34875 rows to 'dim_geo'...
The file was successfully loaded to 'dim_geo' in MySQL.
Opening SQL connection + loading file'dim_geo' took:  6.22sec


In [45]:
#-----------------------------------------------------------------------------
# 3. LOADING sites to SQL
#-----------------------------------------------------------------------------

sites_clean_df = fn.clean_sites_file("file2")

#replace 700/800/1800/2100 5G by fake 5G
def fake_5g(row):
    if row["site_5g_700_m_hz"] == 1 or row["site_5g_800_m_hz"] == 1 or row["site_5g_1800_m_hz"] == 1 or row["site_5g_2100_m_hz"] == 1:
        return 1
    else:
        return 0
    

sites_clean_df["fake_5g"] = sites_clean_df.apply(fake_5g, axis=1)

#define the sites table to upload to sql server
columns_to_exclude = ["code_op", "nom_op", "nom_reg", "nom_dep", "insee_dep","nom_com", 
                     "site_5g_800_m_hz", "site_5g_700_m_hz", "site_5g_1800_m_hz","site_5g_2100_m_hz" ]
sites_to_sql = sites_clean_df.drop(columns=columns_to_exclude)

fn.load_to_sql(file_to_load=sites_to_sql, sql_table_name="sites")

File 'insee_sites_clean.csv' is successfully saved to 'data/clean' folder.
Connected to SQL DB: mobile_5g_network_fr
Uploading 28275 rows to 'sites'...
The file was successfully loaded to 'sites' in MySQL.
Opening SQL connection + loading file' sites' took:  15.21sec


In [42]:
#-----------------------------------------------------------------------------
# 4. LOADING dim_operator: # dim_operator is directly created in workbench see sql script: "create_dim_operator.sql"
#-----------------------------------------------------------------------------

sites_clean_df = fn.clean_sites_file("file2")

dim_operator = sites_clean_df[["code_op", "nom_op", "operator_id"]].drop_duplicates().sort_values("code_op")

dim_operator

File 'insee_sites_clean.csv' is successfully saved to 'data/clean' folder.


,code_op,nom_op,operator_id
0,20801,Orange,1
31688,20810,SFR,2
89439,20815,Free Mobile,3
60383,20820,Bouygues Telecom,4


In [29]:
#-----------------------------------------------------------------------------
# 5. LOADING dim_protocol: # dim_protocol is directly created in workbench see sql script: "create_dim_protocol.sql"
#-----------------------------------------------------------------------------
clean_qos_df = fn.clean_qos_df("file1")

dim_protocol = clean_qos_df[["protocol_id", "protocole"]].drop_duplicates()
dim_protocol   


Number of completeley empty colunms is: 74
Number of irrelevant columns is: 6

number of columns before cleaning:  103

number of columns after cleaning:  23
File '5G_qos_clean.csv' is successfully saved to 'data/clean' folder.


,protocol_id,protocole
0,1,WEB
80,2,STREAM
84,3,DOWNLOAD
88,4,UPLOAD


In [40]:
#-----------------------------------------------------------------------------
# 6. LOADING QOS_5G 
#-----------------------------------------------------------------------------

#qos df
clean_qos_df = fn.clean_qos_df("file1")

#define qos table to upload to SQL
qos_to_sql = clean_qos_df.drop(columns = ['protocole', 'operator'])

#Upload qos table
fn.load_to_sql(file_to_load=qos_to_sql, sql_table_name="fact_qos_measurements")


File '5G_qos_clean.csv' is successfully saved to 'data/clean' folder.
Connected to SQL DB: mobile_5g_network_fr
Uploading 299340 rows to 'fact_qos_measurements'...
The file was successfully loaded to 'fact_qos_measurements' in MySQL.
Opening SQL connection + loading file' fact_qos_measurements' took:  198.12sec


In [42]:
qos_to_sql.shape

(299340, 24)

In [43]:
#-----------------------------------------------------------------------------
# 7. EXPORTING DENORMALIZED QOS FILE FROM SQL to CSV
#-----------------------------------------------------------------------------

#write the SQL query
query = "SELECT * FROM mobile_5g_network_fr.qos_denormalized"

#read the output of query
denormalized_qos = pd.read_sql(query, engine)

# Make data parsing

denormalized_qos["insee_com"] = denormalized_qos["insee_com"].astype("string")
denormalized_qos["insee_dep"] = denormalized_qos["insee_dep"].astype("string")

# Site counts -> integers
site_cols = [
    "tot_phys_sites",
    "tot_4g_per_op_com",
    "tot_5g_per_op_com",
    "tot_fake_5g_per_op_com",
    "tot_5g_sites"
]

denormalized_qos[site_cols] = (
    denormalized_qos[site_cols]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)

# hour_start is already in HH:MM:SS format
# No conversion needed

file_name = "denormalized_qos.csv"
file_path = os.path.join(clean_folder, file_name)

# Write the exported file into data/clean folder
denormalized_qos.to_csv(
    file_path,
    index=False,
    encoding="utf-8"
)

print(denormalized_qos.shape)

(299340, 31)
